- multi models
    - actor：需要进行完整的前向+后向传播和梯度计算，内存压力最大。
    - ref (Reference Model) 和 rollout：通常只需要进行`前向传播`来计算`log_prob`或生成样本，内存压力较小。
    - critic：也需要训练，内存压力也很大。
- 分布式
    - 不管是 fsdp 还是 DeepSpeed 本质上都是数据并行 （断点调试时就有多少 gpu 进程）
        - ga =  ppo_mini_batch_size // ppo_micro_batch_size_per_gpu

### basics

- 调试时
    - `trainer.val_before_train`: False
        - 不 val 直接进行训练；
    - `rollout.calculate_log_probs`: `True`
        - (vllm)rollout 阶段记录 log_probs，用于对比调试；

### ppo batch size

- `actor_rollout_ref.actor.ppo_mini_batch_size=256`

    ```python
    # verl/workers/fsdp_workers.py
    self.config.actor.ppo_mini_batch_size *= self.config.rollout.n
    self.config.actor.ppo_mini_batch_size //= self.device_mesh.size() // self.ulysses_sequence_parallel_size
    ```

- `actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=32` (每个GPU上的微批次大小，用于梯度累积。这是实际在单个GPU上处理的样本数。)

    ```python
    self.gradient_accumulation = (
        self.config.ppo_mini_batch_size // self.config.ppo_micro_batch_size_per_gpu
    )
    ```

### log prob

```python
# fsdp_workers.py

__init__
    # normalize rollout config
    if self._is_rollout and self.config.rollout.log_prob_micro_batch_size is not None:
        self.config.rollout.log_prob_micro_batch_size //= (
            self.device_mesh.size() // self.ulysses_sequence_parallel_size
        )
        self.config.rollout.log_prob_micro_batch_size_per_gpu = self.config.rollout.log_prob_micro_batch_size
        
    # normalize ref config
    if self._is_ref and self.config.ref.log_prob_micro_batch_size is not None:
        self.config.ref.log_prob_micro_batch_size //= self.device_mesh.size() // self.ulysses_sequence_parallel_size
        self.config.ref.log_prob_micro_batch_size_per_gpu = self.config.ref.log_prob_micro_batch_size


compute_log_prob
    data.meta_info["micro_batch_size"] = self.config.rollout.log_prob_micro_batch_size_per_gpu

-----

# dp_actor.py

compute_log_prob
    
    self.actor_module.eval()
    
    micro_batches = data.split(micro_batch_size)
    for micro_batch in micro_batches:
        model_inputs = {**micro_batch.batch, **micro_batch.non_tensor_batch}
        with torch.no_grad():
            entropy, log_probs = self._forward_micro_batch(
                model_inputs, temperature=temperature, calculate_entropy=calculate_entropy
            )
compute_ref_log_prob
    micro_batch_size = self.config.ref.log_prob_micro_batch_size_per_gpu
```

### max token

```python
if self.config.use_dynamic_bsz:
    max_token_len = self.config.ppo_max_token_len_per_gpu * self.ulysses_sequence_parallel_size
    micro_batches, _ = prepare_dynamic_batch(mini_batch, max_token_len=max_token_len)
```